In [103]:
import torch
import numpy as np

from utils import *

In [104]:
d1 = 10000
d2 = 1000
r = 10
p = 0.01

In [105]:
if torch.cuda.is_available():
    device = 'cuda:0'
else:
    device = 'cpu'

# dataset
dataset = "syn"
print(dataset)

M = load_data_syn(r, d1, d2, device)

# sample 2 entries per row
#observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), 2)
#observed_M = torch.from_numpy(observed_M).float().to(device)
#masks = torch.from_numpy(masks).to(device)

# IID sample
observed_M, masks = get_uniform_masks(M, p)

# observed second-moment matrix
MTM = M.T @ M
cov_observe_M =  observed_M.T @ observed_M
T_masks = 1 * (cov_observe_M!=0)

syn


In [106]:
# True probability weighting
diag_cov = torch.diag( torch.diag(cov_observe_M) )
T_p = ((1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov))
print("relative error of T_p: ", torch.norm(T_p*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

relative error of T_p:  tensor(1.0076, device='cuda:0')


In [107]:
# single value estimated probability weighting
p_hat = (observed_M!=0).sum()/(d1*d2)
T_single_p_hat = ((1.0 / p_hat) * diag_cov + (1.0 / (p_hat**2)) * (cov_observe_M - diag_cov))
print("relative error of T_single_p_hat: ", torch.norm(T_single_p_hat*T_masks - MTM*T_masks)/torch.norm(MTM*T_masks))

relative error of T_single_p_hat:  tensor(1.0065, device='cuda:0')


In [108]:
print(p)
print(p_hat)

0.01
tensor(0.0100, device='cuda:0')


In [109]:
# Inverse estimated probability weighting in matrix
cov_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
p_hat_matrix = cov_observe_count/d1

T = cov_observe_M / p_hat_matrix

# calculate matrix of p_hat and p
p_hat_matrix = (cov_observe_count/d1)
p_matrix = (torch.diag(torch.diag(torch.ones(d2, d2)*(p-p**2))) + torch.ones(d2, d2)*p**2).to(device)

# calculate (p_hat - p)
print("mean value of (p_hat - p) in matrix: ", (p_matrix*T_masks-p_hat_matrix*T_masks).mean()/(T_masks.sum() / d2**2))
# calculate the relative err of T_p_hat
print("relative error of T_p_hat: ", torch.norm(T*T_masks-MTM*T_masks)/torch.norm(MTM*T_masks))

mean value of (p_hat - p) in matrix:  tensor(-5.8312e-05, device='cuda:0')
relative error of T_p_hat:  tensor(0.0813, device='cuda:0')


In [110]:
print(p_hat_matrix)

tensor([[1.0700e-02, 4.0000e-04, 3.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         2.0000e-04],
        [4.0000e-04, 9.3000e-03, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        [3.0000e-04, 1.0000e-04, 7.5000e-03,  ..., 1.0000e-04, 1.0000e-04,
         2.0000e-04],
        ...,
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 8.9000e-03, 2.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 2.0000e-04, 9.7000e-03,
         1.0000e-04],
        [2.0000e-04, 1.0000e-04, 2.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         9.3000e-03]], device='cuda:0')


In [111]:
print(p_matrix)

tensor([[1.0000e-02, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-02, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-02,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-04],
        ...,
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-02, 1.0000e-04,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-02,
         1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04,  ..., 1.0000e-04, 1.0000e-04,
         1.0000e-02]], device='cuda:0')


In [112]:
# calculate Taylor expansion in diagonal elements
ratio = T_masks.sum() / d2**2
diag_cov = torch.diag( torch.diag(cov_observe_M) )

T_p_1_diag = (-1.0 / p**2) * diag_cov
T_p_2_diag = (1.0 / p**3) * diag_cov 
T_p_3_diag = (-1.0 / p**4) * diag_cov 

item_1 = torch.mul(T_p_1_diag,(p_hat_matrix-p_matrix).diag())
item_2 = torch.mul(T_p_2_diag,(p_hat_matrix-p_matrix).diag()**2)
item_3 = torch.mul(T_p_3_diag,(p_hat_matrix-p_matrix).diag()**3)

#mask_err_Tp = T_p[T_masks] - MTM[T_masks]
#print(MTM)
#print(T_p)
#print(T_p_1)
#print(T_p_2)

# the result of each item in the Taylor expansion
#num = (diag_cov !=0).sum()
num = (T_masks!=0).sum()
print("value of 1st-order item in diag: ", (item_1*T_masks).sum()/num)
print("value of 2nd-order item in diag: ", (item_2*T_masks).sum()/num)
print("value of 2th-order item in diag: ", (item_3*T_masks).sum()/num)

value of 1st-order item in diag:  tensor(-0.6099, device='cuda:0')
value of 2nd-order item in diag:  tensor(0.5906, device='cuda:0')
value of 2th-order item in diag:  tensor(-0.0185, device='cuda:0')


In [113]:
# calculate Taylor expansion in off-diag elements
ratio = T_masks.sum() / d2**2
diag_cov = torch.diag( torch.diag(cov_observe_M) )
T_p_1_offdiag = - (2.0 / (p**3)) * (cov_observe_M - diag_cov)
T_p_2_offdiag = (3.0 / (p**4)) * (cov_observe_M - diag_cov)
T_p_3_offdiag =  - (4.0 / (p**5)) * (cov_observe_M - diag_cov)

delta_p_offdiag = (p_hat_matrix-p_matrix).fill_diagonal_(0)
item_1 = torch.mul(T_p_1_offdiag,delta_p_offdiag)
item_2 = torch.mul(T_p_2_offdiag,delta_p_offdiag**2)
item_3 = torch.mul(T_p_3_offdiag,delta_p_offdiag**3)

#mask_err_Tp = T_p[T_masks] - MTM[T_masks]
#print(MTM)
#print(T_p)
#print(T_p_1)
#print(T_p_2)

# the result of each item in the Taylor expansion
#num = T_masks.sum() - (diag_cov !=0).sum()
num = (T_masks!=0).sum()
print("value of 1st-order item: ", (item_1*T_masks).sum()/num)
print("value of 2nd-order item: ", (item_2*T_masks).sum()/num)
print("value of 2th-order item: ", (item_3*T_masks).sum()/num)

value of 1st-order item:  tensor(-1268.2295, device='cuda:0')
value of 2nd-order item:  tensor(37.9406, device='cuda:0')
value of 2th-order item:  tensor(-1.2584, device='cuda:0')
